In [4]:
import cv2
import numpy as np
import easyocr
import re
from collections import defaultdict, deque
from ultralytics import YOLO

In [5]:
model = YOLO("license_plate_best.pt")
reader = easyocr.Reader(['en'], gpu = True)
plate_pattern = re.compile(r"[A-Z]{2}[0-9]{2}[A-Z]{3}$")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
def correct_number_plate(ocr_text):
    mapping_num_to_alpha = {"0":"O","1":"I", "5":"S", "8":"B"}
    mapping_alpha_to_num = {"O":"0","I":"1","Z":"2","S":"5","B":"8","R":"5"}

    ocr_text = ocr_text.upper().replace(" ","")
    if len(ocr_text) !=7 :
        return ""
    
    corrected = []
    for i,ch in enumerate(ocr_text):
        if i<2 or i>=4:
            if ch.isdigit() and ch in mapping_num_to_alpha:
                corrected.append(mapping_num_to_alpha[ch])
            elif ch.isalpha():
                corrected.append(ch)
            else:
                return ""
            
        else:
            if ch.isalpha() and ch in mapping_num_to_alpha:
                corrected.append(mapping_alpha_to_num[ch])
            elif ch.isdigit():
                corrected.append(ch)
            else:
                return ""
    return "".join(corrected)


In [8]:
def recognize_plate(plate_crop):
    if plate_crop.size == 0:
        return ""
        # Preprocess for OCR
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
        
        # Apply Otsu's thresholding correctly (fixing the comma omission)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Resize to enhance resolution for OCR
    plate_resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        
    try:
        # Run EasyOCR with specified allowlist (fixing string syntax and adding missing commas)
        ocr_result = reader.readtext(
            plate_resized, 
            detail=0, 
            allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
        )
        
        if len(ocr_result) > 0:
            candidate = correct_number_plate(ocr_result[0])
            # Validate candidate with provided regex pattern
            if candidate and plate_pattern.match(candidate):
                return candidate
    except:
        pass
    return ""

In [9]:
plate_history = defaultdict(lambda: deque(maxlen=10))
plate_final = {}

def get_box_id(x1, y1, x2, y2):
    # Use rounded coordinates as a pseudo ID
    return f"{int(x1/10)}_{int(y1/10)}_{int(x2/10)}_{int(y2/10)}"

def get_stable_plate(box_id, new_text):
    if new_text:
        plate_history[box_id].append(new_text)

        # Majority vote
        most_common = max(
            set(plate_history[box_id]),
            key=plate_history[box_id].count
        )

        plate_final[box_id] = most_common

    return plate_final.get(box_id, "")

In [10]:
input_video = "input.mp4"
output_video = "output.mp4"
CONF_THRESH = 0.3

cap = cv2.VideoCapture(input_video)

# 2. Extract properties and set up the VideoWriter object
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(3))   # Property 3
height = int(cap.get(4)) # Property 4

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))



In [ ]:
# Ensure files/streams are successfully opened
if not cap.isOpened():
    print("Error: Could not open the input video.")
    exit()

# 3. Main processing loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # End of the video file reached

    # --- YOUR PROCESSING OR OBJECT DETECTION CODE HERE ---
    # Example pseudo-logic for utilizing your confidence threshold:
    # 
    results = model(frame)
    for r in results:
        if det.confidence >= CONF_THRESH:
            draw_bounding_box(frame, det)
    # ------------------------------------------------------

    # 4. Write the processed frame to the output file
    out.write(frame)

    # Optional: Display the frame as it processes
    cv2.imshow('Processing Video', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 5. Clean up and release system resources
cap.release()
out.release()
cv2.destroyAllWindows()